# Training report — Two-Phase Exchangeable-Context CCM–FMASAC

Run: `runs/retrain_v3_phys` (Phase 1: 15 000 iters; Phase 2: 800 iters).

This notebook shows **(1)** how training was configured and how it converged,
**(2)** the evaluation metrics with their definitions —
*detection accuracy*, *detection delay*, *reward*, *recovery time*,
*robustness degradation* — and **(3)** per-episode plots of the true change
point vs the detector's detection time.

In [1]:
import sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if (ROOT / "method").exists():
    sys.path.insert(0, str(ROOT))
RUN = ROOT / "runs" / "retrain_v3_phys"
P1_CKPT   = RUN / "phase1_checkpoint_14999.pt"
DETECTOR  = RUN / "phase2_detector.pt"
CFG       = "world_config_5v.yaml"
CHANGE_STEP = 60          # true change point used in every change episode
HORIZON     = 200
plt.rcParams["figure.dpi"] = 110
print("run dir:", RUN)
print("phase-1 checkpoint:", P1_CKPT.name, "| detector:", DETECTOR.name)

run dir: /home/abodeh/thesis/runs/retrain_v3_phys
phase-1 checkpoint: phase1_checkpoint_14999.pt | detector: phase2_detector.pt


## 1  How training was configured

In [2]:
from method.training.phase1 import Phase1Config
from method.training.phase2 import Phase2Config
from method.training.regime import DEFAULT_MODES
from utils.scenario import Scenario

print("train.py args:", json.loads((RUN / "args.json").read_text()))
print()
p1 = Phase1Config()
for k in ["n_envs","horizon","nu","kappa","rho_dim","gamma","lambda_cpc","lambda_cov",
          "p_sw","mass_range","friction_range","n_cpc_negatives","cpc_neg_guard",
          "log_alpha_exp_min","batch_size","lr"]:
    print(f"  Phase1Config.{k:20s} = {getattr(p1,k)}")
print()
p2 = Phase2Config()
for k,v in vars(p2).items():
    print(f"  Phase2Config.{k:16s} = {v}")
print()
print("Dynamics regimes (DEFAULT_MODES): mixture the meta-training task is drawn from")
for m in DEFAULT_MODES:
    print(f"  {m.name:7s}  mass {m.mass_range}   linear_friction {m.friction_range}")
sc = Scenario(config_file=CFG)
print()
print(f"Environment physics:  drag={sc.drag}  u_multiplier={sc.agent_u_multiplier}  "
      f"max_speed={sc.agent_max_speed}  shaping_weight(train)=0.3")
print(f"Task: {len(sc.map.survivals)} victims, {len(sc.map.agents)} agents  "
      f"(victims > 2x agents; rescue_range={sc.rescue_range})")

train.py args: {'phase1_iters': 15000, 'phase2_iters': 800, 'n_envs': 8, 'horizon': 96, 'checkpoint_every': 500, 'log_every': 50, 'config_file': 'world_config_5v.yaml', 'shaping_weight': 0.3, 'seed': 0, 'out_dir': 'runs/retrain_v3_phys', 'resume': None}

  Phase1Config.n_envs               = 8
  Phase1Config.horizon              = 32
  Phase1Config.nu                   = 1.0
  Phase1Config.kappa                = 0.2
  Phase1Config.rho_dim              = 4
  Phase1Config.gamma                = 0.99
  Phase1Config.lambda_cpc           = 1.0
  Phase1Config.lambda_cov           = 0.001
  Phase1Config.p_sw                 = 0.05
  Phase1Config.mass_range           = (0.65, 1.75)
  Phase1Config.friction_range       = (0.0, 0.45)
  Phase1Config.n_cpc_negatives      = 16
  Phase1Config.cpc_neg_guard        = 0.15
  Phase1Config.log_alpha_exp_min    = -3.0
  Phase1Config.batch_size           = 64
  Phase1Config.lr                   = 0.0003

  Phase2Config.d_model          = 64
  Phase2Config.n

### What changed for this run (vs. the earlier degenerate runs)

| Layer | Change | Why |
|---|---|---|
| Environment | `drag` 0.25→0.02, `u_multiplier` 1→4, `max_speed`=1.5 | agents were overdamped (speed ~0.07 u/step): could not cross the arena and a regime change was invisible in the transitions |
| Regimes | `DEFAULT_MODES` = icy / normal / heavy, separated in mass **and** friction | the old modes were near-identical dynamics |
| Exploration | floor `log_alpha_exp` at −3.0 | entropy coefficient had collapsed to ~0.01 (deterministic, no coverage) |
| Reward | potential-based shaping toward nearest unrescued victim (w=0.3) | bare reward too sparse to ever reach a victim |
| Detector (Phase 2) | per-step **BCE anchor** on the "has-switched" label; `λ_FA` 2→0.5; 35 % no-change episodes; uniform switch time | the InDiD delay/FA loss alone collapses to "never fire" |
| Deployment | reset fires only after `p≥threshold` for 3 consecutive steps | kills the free-running detector sawtooth |

## 2  How training converged

In [3]:
d1 = pd.read_csv(RUN / "phase1_log.csv")
d2 = pd.read_csv(RUN / "phase2_log.csv")
sm = lambda s, k=101: s.rolling(k, center=True, min_periods=1).mean()

fig, ax = plt.subplots(2, 3, figsize=(15, 7.5))
ax[0,0].plot(d1.iter, d1.episode_return_agent0, lw=.4, alpha=.35)
ax[0,0].plot(d1.iter, sm(d1.episode_return_agent0), lw=2)
ax[0,0].set_title("Phase 1: rollout return (per episode)"); ax[0,0].axhline(0, color="k", lw=.5)
ax[0,1].plot(d1.iter, sm(d1.l_elbo)); ax[0,1].set_title("Phase 1: L_elbo (encoder recon+KL)"); ax[0,1].set_yscale("log")
ax[0,2].plot(d1.iter, sm(d1.l_cpc)); ax[0,2].set_title("Phase 1: L_cpc (contrastive)"); ax[0,2].set_yscale("log")
ax[1,0].plot(d1.iter, sm(d1.exp_critic_loss), label="exploration")
ax[1,0].plot(d1.iter, sm(d1.exe_critic_loss), label="execution")
ax[1,0].set_title("Phase 1: critic losses"); ax[1,0].set_yscale("log"); ax[1,0].legend()
ax[1,1].plot(d1.iter, sm(d1.alpha_exp_mean)); ax[1,1].set_title("Phase 1: alpha_exp (exploration entropy coeff)")
ax[1,1].axhline(np.exp(-3.0), color="r", ls="--", lw=1, label="floor exp(-3)"); ax[1,1].legend()
ax[1,2].plot(d2.iter, sm(d2.l_cpd, 51)); ax[1,2].set_title("Phase 2: L_cpd (detector loss)")
for a in ax.flat: a.set_xlabel("iteration")
fig.tight_layout(); plt.show()

print(f"return:  start {sm(d1.episode_return_agent0).iloc[:200].mean():+.1f}"
      f"  ->  end {sm(d1.episode_return_agent0).iloc[-200:].mean():+.1f}   (max {d1.episode_return_agent0.max():+.1f})")
print(f"L_elbo:  {d1.l_elbo.iloc[:50].mean():.0f}  ->  {d1.l_elbo.iloc[-200:].mean():.1f}")
print(f"L_cpc :  peak {d1.l_cpc.max():.0f}  ->  end {d1.l_cpc.iloc[-200:].mean():.1f}")

return:  start -4.4  ->  end +13.0   (max +35.3)
L_elbo:  4278  ->  47.9
L_cpc :  peak 199  ->  end 74.3


## 3  Metric definitions

In [4]:
from method import metrics as M
import inspect
for fn in [M.detection_metrics, M.recovery_time, M.degradation]:
    print("="*70); print(fn.__name__); print("-"*70)
    print(inspect.getdoc(fn)); print()
print("="*70)
print("accuracy (detection): (TP + TN) / (TP + TN + FP + FN)  -- thesis 'detection = TP+TN'")
print("detection delay      : steps from true change to first flag within W; undefined if missed")
print("reward               : per-episode team return; reported as IQM with a 95% bootstrap CI")
print("recovery time        : steps after the change until the moving-avg return re-enters +/-10% of pre-change level")
print("robustness degradation: nominal_return - attacked_return (abs & relative)")

detection_metrics
----------------------------------------------------------------------
One episode with a single true change at `true_change` (or None for a
no-change episode). `reset_steps` = sorted list of steps at which the
detector fired a reset for the agent under test.

Returns dict with tp, fp, fn, tn, delay (nan if no detection).

recovery_time
----------------------------------------------------------------------
`step_rewards` = per-step team reward (sum over agents), length ~horizon.
Returns (recovery_steps, censored): steps after `true_change` until the
`window`-step moving average first re-enters +/- `band` of the pre-change
`window`-step average; censored=True (and recovery_steps=horizon-true_change)
if it never does within the episode.

degradation
----------------------------------------------------------------------
Absolute and relative drop. `*_return` are scalars (e.g. IQMs).

accuracy (detection): (TP + TN) / (TP + TN + FP + FN)  -- thesis 'detection = TP+TN'
det

## 4  Change-point detection — accuracy and delay

In [5]:
from method.io import load_model
from method.training.meta_test import MetaTestRunner, MetaTestConfig, RegimeChangeEvent, AdversarialAttackEvent
from method.training.regime import Regime
from method.viz import _victim_state

trainer, detector = load_model(str(P1_CKPT), str(DETECTOR))
factory = lambda: Scenario(config_file=CFG)
NA = trainer.n_agents
NORMAL, HEAVY = Regime(1.10, 0.16), Regime(1.52, 0.36)   # two DEFAULT_MODES centroids
N_EP = 16

def episode(kind, seed, thr=0.5, persist=3):
    torch.manual_seed(seed)
    ev = [RegimeChangeEvent(0, i, NORMAL) for i in range(NA)]
    tc = None
    if kind in ("change", "both"):
        ev.append(RegimeChangeEvent(CHANGE_STEP, 0, HEAVY)); tc = CHANGE_STEP
    if kind in ("attack", "both"):
        ev.append(AdversarialAttackEvent(CHANGE_STEP, 0, True, True))
    r = MetaTestRunner(trainer, detector, factory, MetaTestConfig(threshold_C=thr, trigger_persistence=persist))
    r.schedule(ev); r.reset_state()
    p, resets, rew, mode_exec = [], [], [], []
    for t in range(HORIZON):
        s = r.step(render=False)
        p.append(float(s.detector_p[0] if s.detector_p[0] is not None else 0.0))
        if s.reset_fired[0]: resets.append(t)
        rew.append(float(np.sum(s.rewards)))
        mode_exec.append(s.mode[0])
        _, _, done = _victim_state(r.env)
        if done: break
    scen = r.env.scenario
    rescued = int(sum(bool(sv.rescued[0]) for sv in scen._survivals))
    return dict(kind=kind, tc=tc, p=np.array(p), resets=resets, step_rewards=np.array(rew),
                steps=len(p), total_return=float(np.sum(rew)), rescued=rescued)

chg = [episode("change", 1000+i) for i in range(N_EP)]
noc = [episode("nominal", 5000+i) for i in range(N_EP)]
print(f"ran {len(chg)} change + {len(noc)} no-change episodes")
print(f"detector_p:  change pre {np.mean([e['p'][:CHANGE_STEP].mean() for e in chg]):.3f}"
      f"  post {np.mean([e['p'][CHANGE_STEP:].mean() for e in chg]):.3f}"
      f"  |  no-change {np.mean([e['p'].mean() for e in noc]):.3f}")

ran 16 change + 16 no-change episodes
detector_p:  change pre 0.000  post 0.000  |  no-change 0.000


In [6]:
rows = []
for thr in [0.3, 0.5, 0.7]:
    for persist in [1, 3]:
        det = [M.detection_metrics(
                 [k for k in range(len(e['p'])) if k+1>=persist and np.all(e['p'][k+1-persist:k+1] > thr) and k>=CHANGE_STEP],
                 e['tc'], e['steps']) for e in chg]
        # rebuild TP/FN honestly from thresholded+persistence triggers
        TP=FN=0; delays=[]
        for e in chg:
            trig=[k for k in range(len(e['p'])) if k+1>=persist and np.all(e['p'][k+1-persist:k+1] > thr)]
            post=[k for k in trig if CHANGE_STEP<=k<=CHANGE_STEP+10]
            if post: TP+=1; delays.append(post[0]-CHANGE_STEP)
            else: FN+=1
        TN=sum(1 for e in noc if not [k for k in range(len(e['p'])) if k+1>=persist and np.all(e['p'][k+1-persist:k+1] > thr)])
        FP=len(noc)-TN
        acc=(TP+TN)/(TP+TN+FP+FN)
        rows.append(dict(threshold=thr, persistence=persist, accuracy=round(acc,3),
                         TPR=round(TP/(TP+FN),3), TNR=round(TN/len(noc),3),
                         delay_median=(np.median(delays) if delays else np.nan),
                         delay_mean=(round(np.mean(delays),2) if delays else np.nan),
                         n_detected=len(delays)))
tbl = pd.DataFrame(rows); display(tbl)
if tbl.TPR.max()==0:
    print("\n>>> detector never fires on a real change: p is flat ~0. "
          "Detection accuracy is at chance (0.5) and detection delay is undefined.")

,threshold,persistence,accuracy,TPR,TNR,delay_median,delay_mean,n_detected
0,0.3,1,0.5,0.0,1.0,NaN,NaN,0
1,0.3,3,0.5,0.0,1.0,NaN,NaN,0
2,0.5,1,0.5,0.0,1.0,NaN,NaN,0
3,0.5,3,0.5,0.0,1.0,NaN,NaN,0
4,0.7,1,0.5,0.0,1.0,NaN,NaN,0
5,0.7,3,0.5,0.0,1.0,NaN,NaN,0



>>> detector never fires on a real change: p is flat ~0. Detection accuracy is at chance (0.5) and detection delay is undefined.


### 4b  True change point vs detection time (per episode)

Blue = detector output `p_t`.  Red line = the **true change point** (t = 60).
Green dotted = **detection time** (first persistence-triggered flag ≥ t_c); the
gap to the red line is the **detection delay**.  Grey (right axis) = *steps since
last reset* — the "sawtooth": it ramps every step and drops to 0 on a reset.
The orange line is a model-free **code-surprise** reference (rolling z-distance of
`c_{i,t}` from its recent mean) — this is the signal that IS present in the codes.

In [7]:
@torch.no_grad()
def code_surprise(seed, win=12):
    torch.manual_seed(seed)
    from method.training.regime import apply_regime
    env = trainer.env
    for ag in env.agents: apply_regime(ag, NORMAL)
    obs = env.reset(); buf=[]; out=[]
    for t in range(1, HORIZON+1):
        if t == CHANGE_STEP:
            for ag in env.agents: apply_regime(ag, HEAVY)
        acts=[trainer.exploration_policies[i].act(obs[i])[0] for i in range(NA)]
        nobs,rew,dn,_=env.step(acts)
        cond=torch.cat([torch.stack(obs,1),torch.stack(acts,1)],-1)
        y=torch.cat([torch.stack(rew,1).unsqueeze(-1),torch.stack(nobs,1)],-1)
        c,_=trainer.flow(y,cond); c=c[0,0].numpy()
        if len(buf)>=win:
            mu=np.mean(buf[-win:],0); sd=np.std(buf[-win:],0)+1e-6
            out.append(float(np.linalg.norm((c-mu)/sd)/np.sqrt(len(c))))
        else: out.append(0.0)
        buf.append(c); obs=nobs
        if bool(dn.all()): break
    return np.array(out)

fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
for j, (ax, e) in enumerate(zip(axes, chg[:3])):
    saw=np.zeros(e['steps']); c=0
    for k in range(e['steps']):
        c = 0 if k in e['resets'] else c+1
        saw[k]=c
    surp = code_surprise(1000 + j)
    a2 = ax.twinx(); a2.fill_between(range(e['steps']), saw, color="0.85"); a2.set_ylabel("steps since\nreset", color="0.5", fontsize=8)
    ax.plot(e['p'], color="C0", lw=1.4, label="detector $p_t$", zorder=5)
    ax.plot(surp[:e['steps']]/max(surp.max(),1e-6), color="C1", lw=1.1, alpha=.8, label="code-surprise (norm.)")
    ax.axhline(0.5, color="k", ls="--", lw=.7)
    ax.axvline(CHANGE_STEP, color="C3", lw=2.2, label="true change point")
    post=[k for k in e['resets'] if CHANGE_STEP<=k<=CHANGE_STEP+10]
    if post:
        ax.axvline(post[0], color="C2", ls=":", lw=2.2, label=f"detection (delay {post[0]-CHANGE_STEP})")
    else:
        ax.text(CHANGE_STEP+12, .6, "no detection", color="C3", fontsize=9)
    ax.set_ylim(-.05, 1.08); ax.set_ylabel("$p_t$ / surprise")
    ax.set_title(f"episode (rescued {e['rescued']}/5, {e['steps']} steps)", loc="left", fontsize=9)
    ax.set_zorder(a2.get_zorder()+1); ax.patch.set_visible(False)
axes[0].legend(fontsize=7, loc="upper left", ncol=2); axes[-1].set_xlabel("step")
fig.suptitle("True change point vs detection time"); fig.tight_layout(rect=[0,0,1,.97]); plt.show()

## 5  Reward and recovery time

In [8]:
atk = [episode("attack", 7000+i) for i in range(N_EP)]

def rep(label, eps):
    r = np.array([e['total_return'] for e in eps])
    lo, hi = M.bootstrap_ci(r)
    return dict(condition=label, n=len(r), return_IQM=round(M.iqm(r),2),
               ci_low=round(lo,2), ci_high=round(hi,2),
               rescued_mean=round(np.mean([e['rescued'] for e in eps]),2),
               steps_mean=round(np.mean([e['steps'] for e in eps]),1))
rew_tbl = pd.DataFrame([rep("nominal", noc), rep("change @60", chg), rep("attack @60", atk)])
display(rew_tbl)

# recovery time after the change (change episodes only)
rec = [M.recovery_time(e['step_rewards'], e['tc'], e['steps']) for e in chg]
agg = M.aggregate_recovery(rec)
print("recovery time after the change:", {k: round(v,2) if isinstance(v,float) else v for k,v in agg.items()})

# robustness degradation: nominal vs attacked
deg = M.degradation(M.iqm([e['total_return'] for e in noc]), M.iqm([e['total_return'] for e in atk]))
print("degradation under adversarial attack:", {k: round(v,3) for k,v in deg.items()})

,condition,n,return_IQM,ci_low,ci_high,rescued_mean,steps_mean
0,nominal,16,-27.37,-37.82,-14.72,0.94,200.0
1,change @60,16,-29.56,-37.93,-22.88,0.69,200.0
2,attack @60,16,-40.00,-40.00,-33.70,0.38,200.0


recovery time after the change: {'recovery_mean': 0.0, 'recovery_median': 0.0, 'censored_fraction': 0.06, 'n_recovered': 15, 'n_episodes': 16}
degradation under adversarial attack: {'degradation_abs': 12.63, 'degradation_rel': 0.461}


In [9]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for i,(lab,eps) in enumerate([("nominal",noc),("change",chg),("attack",atk)]):
    r=[e['total_return'] for e in eps]
    ax[0].scatter(np.full(len(r), i)+np.random.uniform(-.08,.08,len(r)), r, s=18, alpha=.6)
    ax[0].hlines(M.iqm(r), i-.2, i+.2, color="k", lw=2)
ax[0].set_xticks([0,1,2]); ax[0].set_xticklabels(["nominal","change","attack"])
ax[0].set_ylabel("episode team return"); ax[0].set_title("Reward by condition (bar = IQM)")

mean_curve = np.nanmean([np.pad(e['step_rewards'][:150].astype(float), (0,max(0,150-len(e['step_rewards']))), constant_values=np.nan)
                         for e in chg], axis=0)
ax[1].plot(mean_curve); ax[1].axvline(CHANGE_STEP, color="C3", lw=2, label="change")
ax[1].set_title("Mean step reward around the change (change episodes)")
ax[1].set_xlabel("step"); ax[1].set_ylabel("team reward/step"); ax[1].legend()
fig.tight_layout(); plt.show()

## 6  One rescue episode (qualitative)

In [10]:
torch.manual_seed(2024)
r = MetaTestRunner(trainer, detector, factory, MetaTestConfig(threshold_C=0.5, trigger_persistence=3))
r.schedule([RegimeChangeEvent(0,i,NORMAL) for i in range(NA)] + [RegimeChangeEvent(CHANGE_STEP,0,HEAVY)])
r.reset_state()
H_, R_, M_, frames = [], [], [], []
for t in range(HORIZON):
    s = r.step(render=(t % 20 == 0))
    hs,_,done = _victim_state(r.env)
    H_.append(hs); R_.append(float(np.sum(s.rewards))); M_.append(s.mode[0])
    if s.frame is not None: frames.append((t, s.frame))
    if done: break
scen = r.env.scenario
print(f"episode ended at step {len(H_)}  |  victims rescued: "
      f"{sum(bool(sv.rescued[0]) for sv in scen._survivals)}/5  |  cumulative return {np.sum(R_):.1f}")

fig, ax = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
ax[0].plot(H_); ax[0].set_ylabel("min victim health"); ax[0].axvline(CHANGE_STEP, color="C3", lw=2)
ax[0].set_title("victim health / step reward / agent-0 mode")
ax[1].plot(np.cumsum(R_)); ax[1].set_ylabel("cumulative return"); ax[1].axvline(CHANGE_STEP, color="C3", lw=2)
exec_mask = np.array([m=="execute" for m in M_])
ax[1].fill_between(range(len(M_)), 0, np.cumsum(R_), where=exec_mask, alpha=.12, label="agent-0 in EXECUTE")
ax[1].legend(); ax[1].set_xlabel("step"); fig.tight_layout(); plt.show()

if frames:
    n=len(frames); fig,axs=plt.subplots(1,n,figsize=(3*n,3))
    for a,(tt,fr) in zip(np.atleast_1d(axs), frames):
        a.imshow(fr); a.set_title(f"t={tt}"); a.axis("off")
    fig.tight_layout(); plt.show()

episode ended at step 200  |  victims rescued: 3/5  |  cumulative return -0.4


## 7  Findings

**What works after v3**
- The policy learns the task: rollout return **−9 → +25…35**, episodes terminate early with all 5 victims rescued.
- The regime change is now **observable** in the context codes — linear-probe AUC pre/post ≈ **0.8** (was 0.56), and the *code-surprise* line in §4b spikes at the true change point.

**What does not work yet**
- The **change-point detector is degenerate**: `p_t ≈ 0` on every episode (change and no-change alike), so detection **accuracy is at chance (0.5)**, **TPR = 0**, and **detection delay is undefined** (no flags fire).
- Cause: the InDiD delay/false-alarm loss collapses to "never fire" even with a strong signal present. The Phase-2 fix (a per-step BCE anchor on the has-switched label, `λ_FA` 2→0.5) was implemented and smoke-tested but its retrain was stopped before completion — `runs/retrain_v3_phys/p2_bce/` holds only a partial log, no detector checkpoint.

**Reward / recovery numbers** in §5 are meaningful (the policy is real); the **detection numbers** in §4 document the current degenerate detector, not a working one.